# M07b.2: fine-tune frozen D0 into D1 (v2: epochs 2 and 3 saved)

**Thin launcher.** The logic and settings live in the repo:

    python -m pii_redteam.cli train-d1 --config configs/m07b_train_d1_v2.yaml

`CONFIG` below selects the recipe. The v2 config is the v1 recipe (same seed, data and
hyperparameters) that also saves `d1-epoch2/` and `d1-epoch3/` with their sha256s and
applies the pre-registered, dev-only freeze rule (fewest name-free dev rows flagged,
within 0.01 of D0's dev F1). Set `CONFIG = "configs/m07b_train_d1.yaml"` to re-run v1.
For the whole night in one go (train, choose, freeze), use `M07b_d1_overnight.ipynb`.

**Runtime -> Change runtime type -> T4 GPU, then Run all.** It needs frozen D0 on
Drive at `pii-redteam-rl/d0-20261004-142714/d0-best`. The first epoch's log line gives
the real per-epoch time.

- **Training data:** `d1_aug_v1` minus a name-disjoint 10% hold-out, plus 3 AI4Privacy
  train-side rows per augmentation row, resampled each epoch.
- **Checkpoint selection:** development data only. An epoch must stay within 0.01 of D0's
  dev span-F1; the highest hold-out F1 wins.
- **Reported afterwards:** D0 vs D1 on each `short_eval` slice, separately.
- **Not here:** final-eval and freezing are `M07b_freeze_d1.ipynb` (or the overnight
  notebook).

A disconnect keeps the report and best checkpoint so far on Drive. It does not resume:
re-run to start a new run, which gets its own directory.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
# Change to "main" once feat/m07b-freeze-d1 is merged.
REF = "feat/m07b-freeze-d1"

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Always sync to the target ref: a reused VM may hold a STALE clone.
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "fetch", "-q", "origin", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", REF, f"origin/{REF}"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", REF, REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True
)

import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU."
free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb >= 10, (
    f"GPU has only {free_gb:.1f} GB free - dirty runtime. Runtime -> Disconnect and"
    " delete runtime, then reopen from GitHub and Run all."
)
print(f"GPU free: {free_gb:.1f} GB - OK ({torch.cuda.get_device_name(0)})")
ENV = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src"), PROBE_REPO_COMMIT=commit)
CONFIG = "configs/m07b_train_d1_v2.yaml"  # v1: configs/m07b_train_d1.yaml

In [ ]:
import glob

result = subprocess.run(
    [sys.executable, "-m", "pii_redteam.cli", "train-d1", "--config", CONFIG],
    cwd=REPO_DIR,
    env=ENV,
)
print(f"\ntrain-d1 exit code: {result.returncode}")
if result.returncode != 0:
    print("^^ the traceback above is the real error - paste it back to me")
PREFIX = "d1v2" if CONFIG.endswith("_v2.yaml") else "d1"
RUN_DIR = sorted(glob.glob(os.path.join(DRIVE_ROOT, f"{PREFIX}-2*")))[-1]
print("latest run dir:", RUN_DIR)

## Final-eval no-regression check for one candidate (off by default)

Prefer `M07b_freeze_d1.ipynb`, which scores every candidate and applies all the gates.

This scores the locked final-eval partition, and each candidate gets one run. Leave it off
until you have reviewed the results above and settled the gates in
`configs/m07b_train_d1.yaml` (D07). To run it later, set `RUN_FINAL_EVAL = True` and run the
cell. In a new session, run the setup cell first and set `RUN_DIR` to the run's Drive
directory.

In [ ]:
RUN_FINAL_EVAL = False

if RUN_FINAL_EVAL:
    subprocess.run(
        [
            sys.executable, "-m", "pii_redteam.cli", "d1-final-eval",
            "--config", CONFIG, "--run-dir", RUN_DIR,
            "--candidate", "d1-candidate",  # or d1-epoch2 / d1-epoch3 for a v2 run
        ],
        cwd=REPO_DIR,
        env=ENV,
        check=True,
    )
else:
    print("Final-eval skipped. D1 is not frozen.")

**When the run finishes:** bring back `d1_report.json` from the run directory on Drive. Its
`experiments_entry` goes into `EXPERIMENTS.yaml`. Don't retrain because of the `short_eval`
numbers without recording that decision, because that would turn `short_eval` into
selection data.